# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesador para ÁRBOLES: solo imputación por mediana (no requieren escala).
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ]
)

# Preprocesador para modelos SENSIBLES A LA ESCALA (LogReg, KNN):
# imputación por mediana + estandarización.
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns),
    ]
)

print("Preprocesadores creados.")

Preprocesadores creados.


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si estandarizamos **antes** de la validación cruzada, el `StandardScaler` calcula la media y la desviación usando **todo** el conjunto de entrenamiento, incluidos los pliegues que luego harán de validación. Eso filtra información de los datos de validación hacia el preprocesamiento (**data leakage**) y produce estimaciones de desempeño **optimistas y sesgadas**. Al colocar el `StandardScaler` **dentro del `Pipeline`**, en cada iteración de CV el escalador se ajusta **solo con los datos de entrenamiento del pliegue** y luego transforma el pliegue de validación. Así cada estimación de validación simula correctamente datos no vistos y, además, el pipeline queda listo para producción sin pasos manuales que repetir.

## 3. Modelos candidatos

In [5]:
# Un pipeline por familia: preprocesamiento + estimador
models = {
    "decision_tree": Pipeline(steps=[
        ("prep", tree_preprocessor),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline(steps=[
        ("prep", scaled_preprocessor),
        ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline(steps=[
        ("prep", scaled_preprocessor),
        ("clf", KNeighborsClassifier()),
    ]),
}
print("Modelos candidatos:", list(models.keys()))

Modelos candidatos: ['decision_tree', 'logistic_regression', 'knn']


## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring="accuracy",
        return_train_score=True,
    )
    rows.append({
        "model": name,
        "train_acc_mean": scores["train_score"].mean(),
        "val_acc_mean": scores["test_score"].mean(),
        "val_acc_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_acc_mean", ascending=False).reset_index(drop=True)
initial_comparison.round(4)

,model,train_acc_mean,val_acc_mean,val_acc_std
0,logistic_regression,1.0000,0.9791,0.0277
1,knn,0.9771,0.9507,0.0353
2,decision_tree,1.0000,0.9022,0.0586


## 5. Optimización de hiperparámetros

In [7]:
# Espacios de búsqueda pequeños por modelo
param_grids = {
    "decision_tree": {
        "clf__max_depth": [2, 3, 5, None],
        "clf__min_samples_leaf": [1, 2, 5],
    },
    "logistic_regression": {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__penalty": ["l2"],
    },
    "knn": {
        "clf__n_neighbors": [3, 5, 7, 9],
        "clf__weights": ["uniform", "distance"],
    },
}

searches = {}
tuned_rows = []
for name, pipe in models.items():
    gs = GridSearchCV(
        pipe, param_grids[name],
        cv=cv, scoring="accuracy", n_jobs=-1,
    )
    gs.fit(X_train, y_train)
    searches[name] = gs
    tuned_rows.append({
        "model": name,
        "best_cv_acc": gs.best_score_,
        "best_params": gs.best_params_,
    })

tuned_comparison = pd.DataFrame(tuned_rows).sort_values("best_cv_acc", ascending=False).reset_index(drop=True)
tuned_comparison

/projects/sandbox/si3003-artificial-intelligence/.venv/lib64/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/projects/sandbox/si3003-artificial-intelligence/.venv/lib64/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penal

,model,best_cv_acc,best_params
0,logistic_regression,0.979064,"{'clf__C': 1.0, 'clf__penalty': 'l2'}"
1,knn,0.957882,"{'clf__n_neighbors': 7, 'clf__weights': 'unifo..."
2,decision_tree,0.902463,"{'clf__max_depth': 5, 'clf__min_samples_leaf': 2}"


## 6. Selección y persistencia del campeón

In [8]:
# Campeón = mayor best_score_ de CV
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion_search = searches[champion_name]
champion = champion_search.best_estimator_   # pipeline COMPLETO ya reentrenado en todo el train

print("Campeón:", champion_name)
print("Accuracy CV:", round(champion_search.best_score_, 4))
print("Mejores hiperparámetros:", champion_search.best_params_)

# Guardar el pipeline completo
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# Metadatos de entrenamiento
training_metadata = {
    "champion": champion_name,
    "cv_accuracy": float(champion_search.best_score_),
    "best_params": {k: v for k, v in champion_search.best_params_.items()},
    "cv_folds": 5,
    "random_state": RANDOM_STATE,
    "candidates": list(models.keys()),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)

# Resultados de CV (inicial + tuned) a disco
cv_results = initial_comparison.merge(
    tuned_comparison[["model", "best_cv_acc"]], on="model", how="left"
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
cv_results.round(4)

Campeón: logistic_regression
Accuracy CV: 0.9791
Mejores hiperparámetros: {'clf__C': 1.0, 'clf__penalty': 'l2'}


,model,train_acc_mean,val_acc_mean,val_acc_std,best_cv_acc
0,logistic_regression,1.0000,0.9791,0.0277,0.9791
1,knn,0.9771,0.9507,0.0353,0.9579
2,decision_tree,1.0000,0.9022,0.0586,0.9025


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> El campeón se eligió por el **mayor accuracy de validación cruzada** (`best_score_`), que es la estimación honesta de desempeño en datos no vistos sin tocar el test. Más allá del promedio, consideramos:
> - **Desempeño:** las tres familias alcanzan accuracy de CV muy alta en Wine (el dataset es casi linealmente separable), pero los modelos sobre features estandarizados (regresión logística / KNN) tienden a liderar.
> - **Variabilidad:** se revisó `val_acc_std`; preferimos un modelo con media alta y desviación baja (resultados estables entre pliegues).
> - **Sobreajuste:** comparando `train_acc_mean` vs `val_acc_mean`, el árbol sin podar tiende a `train=1.0` (memoriza), señal de sobreajuste; por eso la búsqueda limita `max_depth`/`min_samples_leaf`.
> - **Complejidad e interpretabilidad:** si dos modelos empatan, un árbol poco profundo o una regresión logística regularizada son preferibles por ser simples e interpretables. La celda anterior imprime el campeón concreto de esta corrida y sus hiperparámetros.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.